# pyNCBIGene: NCBI Gene annotation in parquet

Python companion to the Bioconductor [RNCBIGene](https://github.com/vjcitn/RNCBIGene) package.
All queries run through a persistent [duckdb](https://duckdb.org) connection that reads
Apache Parquet files directly from an NSF Open Storage Network bucket, pushing filters
and column selections to the parquet layer before any data lands in Python.

The return type of `open_ncbi_gene()` is a `duckdb.DuckDBPyRelation` -- a lazy query object.
Chain `.filter()`, `.select()`, and `.df()` (collect to pandas) as needed.

In [1]:
from pyNCBIGene import (
    available_ncbi_parquet,
    ncbi_gene_fields,
    ncbi_parquet_info,
    open_ncbi_gene,
    map_ids_ng,
    join_ncbi_gene,
)
import pandas as pd

## 1. Scope: available resources

Eight parquet files live in the OSN bucket, covering all organisms in NCBI Gene.

In [2]:
available_ncbi_parquet()

['gene2accession.parquet',
 'gene2ensembl.parquet',
 'gene2go.parquet',
 'gene2pubmed.parquet',
 'gene2refseq.parquet',
 'gene_info.parquet',
 'gene_orthologs.parquet',
 'gene_refseq_uniprotkb_collab.parquet']

Record counts -- `COUNT(*)` pushed to duckdb, reads row-group metadata without fetching data.

In [3]:
resources = [r.replace('.parquet', '') for r in available_ncbi_parquet()]
counts = {r: open_ncbi_gene(r).count('*').fetchone()[0] for r in resources}
pd.Series(counts, name='n_rows').sort_index()

gene2accession                  282519101
gene2ensembl                     18069267
gene2go                         122971050
gene2pubmed                      79863715
gene2refseq                     106157713
gene_info                        70797156
gene_orthologs                   18590068
gene_refseq_uniprotkb_collab    124490938
Name: n_rows, dtype: int64

Bucket metadata -- sizes, upload timestamps, NCBI source dates.

In [4]:
ncbi_parquet_info()

,resource,size_bytes,bucket_modified,ncbi_last_modified
0,gene2accession.parquet,3111866681,2026-07-22T19:00:09.796Z,"Wed, 22 Jul 2026 06:04:29 GMT"
1,gene2ensembl.parquet,201322528,2026-07-22T19:00:14.050Z,"Wed, 22 Jul 2026 06:04:48 GMT"
2,gene2go.parquet,550607120,2026-07-22T19:00:31.108Z,"Wed, 22 Jul 2026 06:06:46 GMT"
3,gene2pubmed.parquet,115550070,2026-07-22T19:00:33.657Z,"Wed, 22 Jul 2026 06:07:08 GMT"
4,gene2refseq.parquet,1460763355,2026-07-22T19:01:18.313Z,"Wed, 22 Jul 2026 06:10:14 GMT"
5,gene_info.parquet,951206411,2026-07-22T19:01:51.231Z,"Wed, 22 Jul 2026 06:12:27 GMT"
6,gene_orthologs.parquet,62310042,2026-07-22T19:01:54.029Z,"Wed, 22 Jul 2026 06:14:51 GMT"
7,gene_refseq_uniprotkb_collab.parquet,510820503,2026-07-22T19:02:15.685Z,"Tue, 21 Jul 2026 09:12:44 GMT"


## 2. Lazy remote queries with `open_ncbi_gene()`

`open_ncbi_gene()` returns a lazy `DuckDBPyRelation` backed by a duckdb VIEW over the
remote parquet file.  No data is fetched until `.df()` is called.

In [5]:
tbl = open_ncbi_gene('gene_info', taxid=9606)
type(tbl)

_duckdb.DuckDBPyRelation

Compose a filter + select before collecting -- only matched rows and columns travel the wire.

In [6]:
(
    open_ncbi_gene('gene_info', taxid=9606)
    .filter('"Symbol" IN (\'TP53\', \'ORMDL3\', \'BRCA1\', \'GSDMB\')')
    .select('"Symbol", "GeneID", "map_location", "type_of_gene"')
    .df()
)

,Symbol,GeneID,map_location,type_of_gene
0,TP53,7157,17p13.1,protein-coding
1,GSDMB,55876,17q21.1,protein-coding
2,ORMDL3,94103,17q21.1,protein-coding
3,BRCA1,672,17q21.31,protein-coding


When `taxid` is specified the `#tax_id` column is dropped from the result automatically.
Omitting `taxid` returns a relation spanning all organisms:

In [7]:
# TP53 orthologs across all taxa (returns many rows)
open_ncbi_gene('gene_orthologs').filter('"GeneID" = 7157').df().head(10)

,#tax_id,GeneID,relationship,Other_tax_id,Other_GeneID
0,9606,7157,Ortholog,7918,102690789
1,9606,7157,Ortholog,7936,118235846
2,9606,7157,Ortholog,7938,135263502
3,9606,7157,Ortholog,7955,30590
4,9606,7157,Ortholog,7957,113048774
5,9606,7157,Ortholog,7959,127512451
6,9606,7157,Ortholog,7962,109083614
7,9606,7157,Ortholog,7994,103046742
8,9606,7157,Ortholog,7998,100304476
9,9606,7157,Ortholog,8005,113570174


## 3. Discovering available fields with `ncbi_gene_fields()`

In [8]:
ncbi_gene_fields('gene_info')

,column_name,column_type
0,#tax_id,BIGINT
1,GeneID,BIGINT
2,Symbol,VARCHAR
3,LocusTag,VARCHAR
4,Synonyms,VARCHAR
5,dbXrefs,VARCHAR
6,chromosome,VARCHAR
7,map_location,VARCHAR
8,description,VARCHAR
9,type_of_gene,VARCHAR


In [9]:
ncbi_gene_fields('gene2go')

,column_name,column_type
0,#tax_id,BIGINT
1,GeneID,BIGINT
2,GO_ID,VARCHAR
3,Evidence,VARCHAR
4,Qualifier,VARCHAR
5,GO_term,VARCHAR
6,PubMed,VARCHAR
7,Category,VARCHAR


In [10]:
# programmatic validation
gi_cols = ncbi_gene_fields('gene_info')['column_name'].tolist()
go_cols = ncbi_gene_fields('gene2go')['column_name'].tolist()
print('map_location in gene_info:', 'map_location' in gi_cols)
print('map_location in gene2go:  ', 'map_location' in go_cols)

map_location in gene_info: True
map_location in gene2go:   False


## 4. Identifier mapping with `map_ids_ng()`

All filtering is pushed to duckdb before a single `.df()` fetch.  Returns a dict;
`None` for unrecognised keys.

### Symbol to GeneID

In [11]:
map_ids_ng(['ORMDL3', 'TP53', 'GSDMB', 'XyZZY'], keytype='Symbol',
           column='GeneID', taxid=9606)

{'ORMDL3': 94103, 'TP53': 7157, 'GSDMB': 55876, 'XyZZY': None}

### Symbol to chromosomal map location

In [12]:
map_ids_ng(['ORMDL3', 'TP53', 'GSDMB', 'BRCA1'], keytype='Symbol',
           column='map_location', taxid=9606)

{'ORMDL3': '17q21.1',
 'TP53': '17p13.1',
 'GSDMB': '17q21.1',
 'BRCA1': '17q21.31'}

### Ensembl to Symbol

A JOIN between `gene2ensembl` and `gene_info` runs entirely in duckdb.

In [13]:
map_ids_ng(
    ['ENSG00000073605', 'ENSG00000141510', 'ENSG00000012048'],
    keytype='Ensembl', column='Symbol', taxid=9606
)

{'ENSG00000073605': 'GSDMB',
 'ENSG00000141510': 'TP53',
 'ENSG00000012048': 'BRCA1'}

## 5. Non-human organisms

All resources cover all NCBI organisms.  Switching species is a matter of changing `taxid`.
Here we look up Ensembl gene and transcript identifiers for the mouse (_Mus musculus_,
taxid 10090) gene _Lilrb4a_.

In [14]:
# quick Symbol -> Ensembl gene ID
map_ids_ng(['Lilrb4a'], keytype='Symbol', column='Ensembl', taxid=10090)

{'Lilrb4a': 'ENSMUSG00000112148'}

For the full transcript-level picture, join `gene_info` and `gene2ensembl` in duckdb.
Both relations share the same connection so the join executes before any data is collected.

In [15]:
from pyNCBIGene._state import get_connection

info = (
    open_ncbi_gene('gene_info', taxid=10090)
    .filter('"Symbol" = \'Lilrb4a\'')
    .select('"GeneID", "Symbol"')
)
g2e = (
    open_ncbi_gene('gene2ensembl', taxid=10090)
    .select('"GeneID", "Ensembl_gene_identifier", "Ensembl_rna_identifier", "Ensembl_protein_identifier"')
)

con = get_connection()
con.register('_info', info.df())
con.register('_g2e',  g2e.df())
con.sql(
    'SELECT i."Symbol", g."Ensembl_gene_identifier", '
    'g."Ensembl_rna_identifier", g."Ensembl_protein_identifier" '
    'FROM _info i JOIN _g2e g USING ("GeneID")'
).df()

,Symbol,Ensembl_gene_identifier,Ensembl_rna_identifier,Ensembl_protein_identifier
0,Lilrb4a,ENSMUSG00000062593,ENSMUST00000218123.2,ENSMUSP00000151827.2
1,Lilrb4a,ENSMUSG00000112148,ENSMUST00000078778.5,ENSMUSP00000077833.4
2,Lilrb4a,ENSMUSG00000112148,ENSMUST00000218617.2,-


The result reveals two distinct Ensembl gene models for _Lilrb4a_ -- something
a two-step approach (Symbol -> one gene ID -> transcripts) would miss.

## 6. Joining local data to remote annotation with `join_ncbi_gene()`

`join_ncbi_gene()` validates the join columns against both the local DataFrame and the
remote resource, then executes the join in duckdb.

In [16]:
local_df = pd.DataFrame({'Symbol': ['ORMDL3', 'TP53', 'BRCA1', 'GSDMB', 'XyZZY']})

join_ncbi_gene(local_df, by='Symbol', resource='gene_info', taxid=9606) \
    .select('"Symbol", "GeneID", "map_location", "type_of_gene"') \
    .df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Symbol,GeneID,map_location,type_of_gene
0,BRCA1,672,17q21.31,protein-coding
1,TP53,7157,17p13.1,protein-coding
2,GSDMB,55876,17q21.1,protein-coding
3,ORMDL3,94103,17q21.1,protein-coding
4,XyZZY,<NA>,None,None


Unmatched keys (`XyZZY`) appear with `NaN` in annotation columns because the default
join type is `'left'`.  Use `how='inner'` to drop them.

### GO terms and RefSeq: separate joins required

GO terms and RefSeq IDs are both one-to-many from GeneID.  Joining both resources in a
single query would produce a Cartesian product -- the correct pattern is two independent joins.

In [17]:
local_ids = pd.DataFrame({'GeneID': [94103, 7157, 672]})

# GO annotations
go_df = (
    join_ncbi_gene(local_ids, by='GeneID', resource='gene2go',
                  taxid=9606, how='inner')
    .select('"GeneID", "GO_ID", "GO_term", "Evidence", "Category"')
    .df()
)
print(f'GO rows: {len(go_df)}')
go_df.head()

GO rows: 343


,GeneID,GO_ID,GO_term,Evidence,Category
0,672,GO:0003677,DNA binding,TAS,Function
1,672,GO:0003684,damaged DNA binding,IBA,Function
2,672,GO:0003684,damaged DNA binding,IEA,Function
3,672,GO:0003713,transcription coactivator activity,IDA,Function
4,672,GO:0003713,transcription coactivator activity,IEA,Function


In [18]:
# RefSeq identifiers
refseq_df = (
    join_ncbi_gene(local_ids, by='GeneID', resource='gene2refseq',
                  taxid=9606, how='inner')
    .select('"GeneID", "RNA_nucleotide_accession.version", "protein_accession.version"')
    .filter('"RNA_nucleotide_accession.version" != \'-\'')
    .df()
)
print(f'RefSeq rows: {len(refseq_df)}')
refseq_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

RefSeq rows: 810


,GeneID,RNA_nucleotide_accession.version,protein_accession.version
0,672,NM_001407571.1,NP_001394500.1
1,672,NM_001407571.1,NP_001394500.1
2,672,NM_001407581.1,NP_001394510.1
3,672,NM_001407581.1,NP_001394510.1
4,672,NM_001407582.1,NP_001394511.1


## 7. Local caching with `cache_by_taxon()`

All queries above hit the OSN bucket on every call.  `cache_by_taxon()` performs a
one-time filter of each remote parquet to a single taxon and stores the result locally.
After that, `open_ncbi_gene()` routes to the local file automatically.

Requires `pybiocfilecache`: `pip install pybiocfilecache`

In [19]:
# from pyNCBIGene import cache_by_taxon, taxon_cache_info, clear_taxon_cache

# one-time setup -- filters and writes local parquet for each resource
# cache_by_taxon(9606)        # human: all eight resources (slow)
# cache_by_taxon(9606, resources=['gene_info', 'gene2go'])  # subset

# check what is cached
# taxon_cache_info(9606)

# identical call to open_ncbi_gene -- routed to local file automatically
# open_ncbi_gene('gene_info', taxid=9606).filter(...).df()

# remove live cache and route back to OSN
# clear_taxon_cache(9606)

print('Caching cells are commented out to avoid network/disk costs in this demo.')
print('Uncomment and run interactively after pip install pybiocfilecache.')

Caching cells are commented out to avoid network/disk costs in this demo.
Uncomment and run interactively after pip install pybiocfilecache.


## 8. Reproducibility snapshots with `freeze_taxon_cache()`

A live cache can be overwritten at any time.  `freeze_taxon_cache()` makes a physical
copy under an identifying tag.  Frozen entries survive `clear_taxon_cache()`.
Duplicate tags are rejected unless `force=True`.

In [20]:
# from pyNCBIGene import freeze_taxon_cache

# freeze_taxon_cache(9606, tag='paper_2026_07')  # snapshot live cache

# query the frozen snapshot via freeze_tag=
# open_ncbi_gene('gene_info', taxid=9606, freeze_tag='paper_2026_07') \
#     .filter('"Symbol" = \'TP53\'').df()

# stop with KeyError if tag not found -- fails loudly, not silently
# open_ncbi_gene('gene_info', taxid=9606, freeze_tag='nonexistent')

print('Freeze cells are commented out -- requires prior cache_by_taxon() call.')

Freeze cells are commented out -- requires prior cache_by_taxon() call.


## Session info

In [21]:
import sys, duckdb, pandas
import pyNCBIGene

print(f'Python:      {sys.version}')
print(f'pyNCBIGene:  {pyNCBIGene.__version__}')
print(f'duckdb:      {duckdb.__version__}')
print(f'pandas:      {pandas.__version__}')

Python:      3.12.5 (v3.12.5:ff3bc82f7c9, Aug  7 2024, 05:32:06) [Clang 13.0.0 (clang-1300.0.29.30)]
pyNCBIGene:  0.1.0
duckdb:      1.5.5
pandas:      2.2.3
